<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/healthstack-logo-dark.png" alt="HealthStack logo" width="320" />
</div>

<p align="center"><strong>AFROTECH 2026</strong></p>
<p align="center"><font size="6"><strong>Building Fairness Into the Data Supply Chain</strong></font></p>
<p align="center"><strong>Instructor: Waverly Rose Brim</strong></p>

---

<p align="center"><font size="6"><strong>The Impact of One Lab Value</strong></font></p>

<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/consider-one-patient.jpg" alt="Portrait of a middle-aged Black man seated in an exam room before clinical vitals are taken." width="480" />
  <p><em>A patient at the start of a routine visit.</em></p>
</div>

---

## Start here
This notebook answers two questions:
1. What is eGFR?
2. What changes when the equation changes?

**Workflow:** predict -> run -> inspect -> interpret.

## Colab setup
Skip this cell if you are running locally.

In [ ]:
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps", "--no-cache-dir",
        "data-supply-chain-workshop==0.1.2",
    ])
else:
    import data_supply_chain

### **One patient, two equations**

Consider one patient: 50 years old, male, Black, with a serum creatinine of 1.5 mg/dL.

In [ ]:
# Demo patient, take this opportunity to review the output
 
from IPython.display import Image, clear_output, display

clear_output(wait=True)

from data_supply_chain.bootstrap import cold_start
from data_supply_chain.egfr import (
    DEMO_PATIENT,
    ckd_stage,
    egfr_2009_race,
    egfr_2021_racefree,
)
from data_supply_chain.figures import egfr_gates_png

S = cold_start()
print(f"Cohort loaded: {len(S.df):,} adults · source={S.source} · real={S.is_real}\n")

p = DEMO_PATIENT
egfr_2009 = float(egfr_2009_race(p["scr"], p["age"], p["female"], p["black"]))
egfr_2021 = float(egfr_2021_racefree(p["scr"], p["age"], p["female"]))
stage_2009 = str(ckd_stage(egfr_2009)[0])
stage_2021 = str(ckd_stage(egfr_2021)[0])

delta_check = egfr_2021 - egfr_2009
crossed_60_check = (egfr_2009 >= 60) and (egfr_2021 < 60)

print(f"{p['age']}-year-old {'female' if p['female'] else 'male'} patient")
print(f"Recorded race: {'Black' if p['black'] else 'Non-Black'}")
print(f"Serum creatinine: {p['scr']:.1f} mg/dL\n")
print("Equation                        eGFR      CKD stage")
print("-" * 52)
print(f"CKD-EPI 2009 (with race)      {egfr_2009:6.1f}     {stage_2009}")
print(f"CKD-EPI 2021 (race-free)      {egfr_2021:6.1f}     {stage_2021}")
print(f"\nChange: {delta_check:+.1f} mL/min/1.73 m^2")
print(f"Crossed below 60 threshold: {crossed_60_check}")

display(Image(data=egfr_gates_png(egfr_2009, egfr_2021), format="png"))

### **Attendee challenge: change the creatinine**

Curious? Investigate how changing the creatinine value moves a patient across different thresholds, or doesn't.

Edit `challenge_scr` in the next cell (try 1.2, 1.8, or 2.2) and re-run.

The table and the figure both update.

In [ ]:
# Your turn, investigate.

from IPython.display import Image, clear_output, display

clear_output(wait=True)

# Edit this line; try 1.2, 1.8, or 2.2
challenge_scr = 1.2

ch_2009 = float(egfr_2009_race(challenge_scr, p["age"], p["female"], p["black"]))
ch_2021 = float(egfr_2021_racefree(challenge_scr, p["age"], p["female"]))
ch_stage_2009 = str(ckd_stage(ch_2009)[0])
ch_stage_2021 = str(ckd_stage(ch_2021)[0])

print(f"{p['age']}-year-old {'female' if p['female'] else 'male'} patient")
print(f"Recorded race: {'Black' if p['black'] else 'Non-Black'}")
print(f"Serum creatinine: {challenge_scr:.1f} mg/dL  (recorded was {p['scr']:.1f})\n")
print("Equation                        eGFR      CKD stage")
print("-" * 52)
print(f"CKD-EPI 2009 (with race)      {ch_2009:6.1f}     {ch_stage_2009}")
print(f"CKD-EPI 2021 (race-free)      {ch_2021:6.1f}     {ch_stage_2021}")
print(f"\nChange: {ch_2021 - ch_2009:+.1f} mL/min/1.73 m^2")
print(f"Crossed below 60 threshold: {(ch_2009 >= 60) and (ch_2021 < 60)}")

display(Image(data=egfr_gates_png(ch_2009, ch_2021), format="png"))

---

## Maintainer cell (safe to skip in workshop)
This writes the findings the website reads, so the numbers on the page stay
synced to this notebook's output.

In [ ]:
from data_supply_chain.findings import Findings
from data_supply_chain.paths import findings_path

f = Findings.open()
f.record("cohort.n", len(S.df))
f.record("act1.demo.age", p["age"])
f.record("act1.demo.scr", p["scr"])
f.record("act1.demo.egfr_2009", round(egfr_2009, 1))
f.record("act1.demo.egfr_2021", round(egfr_2021, 1))
f.record("act1.demo.stage_2009", stage_2009)
f.record("act1.demo.stage_2021", stage_2021)
f.record("act1.demo.egfr_drop", round(egfr_2009 - egfr_2021, 1))

if findings_path() is None:
    print("Outside a repo checkout - findings.json not written (expected in Colab).")
else:
    f.write(state=S)
    print("wrote findings.json")